In [7]:
# This cell was produced with the help of ChatGPT.

import pandas as pd
import numpy as np

def generate_ev_usage_timeseries(start="2025-06-11 00:00", end="2025-06-12 00:00", 
                                 freq="15T", num_evs=3, seed=42):
    np.random.seed(seed)
    
    # Time index
    index = pd.date_range(start=start, end=end, freq=freq, inclusive="left")
    df = pd.DataFrame(index=index)

    # Define time-of-day transition profiles
    def get_state_probabilities(timestamp):
        hour = timestamp.hour
        if 7 <= hour < 9:
            return {"D": 0.8, "S": 0.1, "C": 0.1}  # morning commute
        elif 9 <= hour < 18:
            return {"D": 0.1, "S": 0.9, "C": 0.0}  # work hours (no charging)
        elif 18 <= hour < 20:
            return {"D": 0.2, "S": 0.2, "C": 0.6}  # evening charging
        elif 20 <= hour < 24:
            return {"D": 0.1, "S": 0.2, "C": 0.7}  # evening wind-down
        elif 0 <= hour < 7:
            return {"D": 0.01, "S": 0.0, "C": 0.99}  # night

    # Generate usage for each EV
    last_state = None
    
    for i in range(num_evs):
        ev_col = []
        for ts in index:
            probs = get_state_probabilities(ts)

            # A EV stays either at charging station - or it does not.
            if last_state == "S":
                probs["S"] += probs["C"] / 2
                probs["D"] += probs["C"] / 2
                probs["C"] = 0
                
            elif last_state == "C":
                probs["C"] += probs["S"] / 2
                probs["D"] += probs["S"] / 2
                probs["S"] = 0

            # Normalize & draw next state
            next_state = np.random.choice(["D", "S", "C"], p=[probs["D"], probs["S"], probs["C"]])
            last_state = next_state
            ev_col.append(next_state)
        df[f"EV{i+1}"] = ev_col

    return df

def remove_single_charging_steps(df):
    """
    Replaces isolated single 'L' (charging) entries with 'S' (stay).
    Assumes 15-minute resolution and categorical values: 'L', 'S', 'D'.
    """
    df_out = df.copy()

    for col in df.columns:
        series = df[col]
        new_series = series.copy()

        # Identify where current is 'L' and neighbors are not 'L'
        for i in range(1, len(series) - 1):
            if series.iat[i] == 'C' and series.iat[i - 1] != 'C' and series.iat[i + 1] != 'C':
                new_series.iat[i] = 'S'

        df_out[col] = new_series

    return df_out

# Example usage
usage_df = generate_ev_usage_timeseries(num_evs=3)
usage_df = remove_single_charging_steps(usage_df)  # First 12 hours

In [8]:
usage_df

,EV1,EV2,EV3
2025-06-11 00:00:00,C,C,C
2025-06-11 00:15:00,C,C,C
2025-06-11 00:30:00,C,C,C
2025-06-11 00:45:00,C,C,C
2025-06-11 01:00:00,C,C,C
...,...,...,...
2025-06-11 22:45:00,C,C,D
2025-06-11 23:00:00,C,C,S
2025-06-11 23:15:00,C,C,D
2025-06-11 23:30:00,C,D,C
